## PDF to Chroma Pipeline
This notebook loads a PDF, splits it into chunks, stores the chunks in Chroma, and runs a retrieval query.

In [1]:
import os
from pathlib import Path

from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings


# Ollama embedding model
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

/tmp/ipykernel_225131/2866186121.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [10]:
collection_name = "rag-pipeline"
project_root = Path.cwd()
persists_directory = project_root / "chroma_langchain_db"
project_root

PosixPath('/home/bs/gen-ai/advanced-rag-campusX/4.vector-stores')

In [4]:
pdf_path = project_root / "documents" / "beyond-chatbots-ai-agents-next-real-shift.pdf"


## Add Small Display Helpers

In [5]:
def preview_text(text, limit=120):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_documents(title, docs):
    """Print retrieved documents using page metadata and a text preview."""
    print(title)
    for index, doc in enumerate(docs, start=1):
        print(f"{index}. page={doc.metadata.get('page')} | source={doc.metadata.get('source')}")
        print(f"   content={doc.page_content}")
    print()

## laod the pdf

In [6]:
loader = PyPDFLoader(str(pdf_path))
docs = loader.load()
print(f"Total pages loaded: {len(docs)}")

Total pages loaded: 6


In [7]:
docs[0].page_content

'Page 1\n Beyond Chatbots: Why AI Agents Feel Like the\n Next Real Shift\nA practical long-form blog on planning, memory, tools, and retrieval in modern AI systems\nBy Editorial Desk\nThe moment AI stopped feeling like a demo\nFor a long time, the most common experience with AI felt theatrical. You typed a question, the model\nanswered in polished language, and for a moment it seemed almost magical. Then the illusion broke. Ask a\nfollow-up that required memory, factual grounding, or a small sequence of actions, and the system often fell\napart. It could sound confident without being connected to anything real. That gap between fluency and\nusefulness is exactly where AI agents enter the picture.\nAn AI agent is interesting not because it sounds human, but because it behaves like software with intent. It\ncan take a goal, figure out what it needs in order to make progress, and work through a sequence of steps\ninstead of improvising a single reply. In the simplest form, that might mean

## split pdf in chunks

In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
)
chunked_docs = text_splitter.split_documents(docs)
print(f"Total chunks created: {len(chunked_docs)}")

Total chunks created: 88


## Store Chunks in Chroma

In [12]:
vector_store = Chroma.from_documents(
    documents=chunked_docs,
    embedding=embeddings,
    collection_name=collection_name,
    persist_directory=str(persists_directory),
)

print(f"Stored {len(chunked_docs)} chunks in the '{collection_name}' collection.")

Stored 88 chunks in the 'rag-pipeline' collection.


## Retrieve Relevant Chunks

In [13]:
query = "How do AI agents use tools and memory?"
query

'How do AI agents use tools and memory?'

In [14]:
query = "How do AI agents use tools and memory?"
query

'How do AI agents use tools and memory?'

In [15]:
results = vector_store.similarity_search(query, k=3)

print(f"Query: {query}\n")
print_documents("Retrieved chunks:", results)

Query: How do AI agents use tools and memory?

Retrieved chunks:
1. page=2 | source=/home/bs/gen-ai/advanced-rag-campusX/4.vector-stores/documents/beyond-chatbots-ai-agents-next-real-shift.pdf
   content=reveals the reasoning surface of the system. The answer no longer feels like a black box. It feels like the
product of a process that can be inspected.
Why tools make agents actually useful
If memory gives an agent context, tools give it reach. A model can describe a search, but a tool can perform
2. page=5 | source=/home/bs/gen-ai/advanced-rag-campusX/4.vector-stores/documents/beyond-chatbots-ai-agents-next-real-shift.pdf
   content=into real work. AI agents are moving in that direction. And the more we build them around retrieval, structure,
and accountability, the more likely they are to stay there.
3. page=0 | source=/home/bs/gen-ai/advanced-rag-campusX/4.vector-stores/documents/beyond-chatbots-ai-agents-next-real-shift.pdf
   content=An AI agent is interesting not because it sound

In [17]:
retrieved_docs = vector_store.similarity_search_with_score(query, k=2)

for doc, score in retrieved_docs:
    print(f"Score: {score:.4f}")
    print(f"Content preview: {doc.page_content}")
    print(f"page_no. {doc.metadata}")
    print()

Score: 0.4470
Content preview: reveals the reasoning surface of the system. The answer no longer feels like a black box. It feels like the
product of a process that can be inspected.
Why tools make agents actually useful
If memory gives an agent context, tools give it reach. A model can describe a search, but a tool can perform
page_no. {'page': 2, 'page_label': '3', 'source': '/home/bs/gen-ai/advanced-rag-campusX/4.vector-stores/documents/beyond-chatbots-ai-agents-next-real-shift.pdf', 'total_pages': 6, 'keywords': '', 'creator': '(unspecified)', 'title': 'Beyond Chatbots: Why AI Agents Feel Like the Next Real Shift', 'producer': 'ReportLab PDF Library - (opensource)', 'moddate': '2026-03-12T20:36:07+05:00', 'creationdate': '2026-03-12T20:36:07+05:00', 'trapped': '/False', 'subject': '(unspecified)', 'author': 'By Editorial Desk'}

Score: 0.4976
Content preview: into real work. AI agents are moving in that direction. And the more we build them around retrieval, structure,
and accounta